# YOLO11n-seg + SimAM+CA + WIoU v3 Box Loss (Nhóm B — B1)

**Cải tiến:** Thay thế CIoU mặc định bằng WIoU v3 cho box regression loss.  
**Lý do:** Dataset annotation thủ công có noise — biên giới BG và WSSV không rõ ràng. WIoU v3 dùng dynamic non-monotonic focusing để giảm gradient từ outlier/noisy labels, tập trung học các mẫu 'mediocre' thay vì bị nhiễu bởi cả easy và hard outliers.  
**Kỳ vọng:** +2-4% mAP50-95 nhờ loại bỏ gradient độc hại từ nhãn sai.  
**Base model:** SimAM+CA (best model từ notebook combined).


## Portable Colab/Kaggle runtime paths

All generated data, cloned dependencies, training runs, and augmented-test outputs use the writable runtime directory selected below.


In [ ]:
from pathlib import Path
import os

if Path("/kaggle/working").exists():
    RUNTIME_ROOT = Path("/kaggle/working")
elif Path("/content").exists():
    RUNTIME_ROOT = Path("/content")
else:
    RUNTIME_ROOT = Path.cwd()

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(RUNTIME_ROOT)
print("Runtime root:", RUNTIME_ROOT)


In [ ]:
# Install dependencies
import importlib.util, subprocess, sys

if importlib.util.find_spec('roboflow') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'])

if importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])

print('Dependencies ready.')

In [ ]:
# ── GPU check ──────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── Download dataset (Roboflow) ────────────────────────────────────────────
import os
from roboflow import Roboflow

ROBOFLOW_API_KEY_DIRECT = ''
ROBOFLOW_WORKSPACE    = 'lets-try-this'
ROBOFLOW_PROJECT      = 'shrimpdishandsegv2'
ROBOFLOW_VERSION      = 1
ROBOFLOW_FORMAT       = 'yolo26'


def get_roboflow_api_key():
    if ROBOFLOW_API_KEY_DIRECT.strip():
        return ROBOFLOW_API_KEY_DIRECT.strip()
    try:
        from google.colab import userdata
        key = userdata.get('ROBOFLOW_API_KEY')
        if key:
            return key
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
        if key:
            return key
    except Exception:
        pass
    return os.environ.get('ROBOFLOW_API_KEY', '').strip()


api_key = get_roboflow_api_key()
if not api_key:
    raise RuntimeError('Missing Roboflow API key.')

rf      = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download(ROBOFLOW_FORMAT)

base_path = str(Path(dataset.location).resolve())
print('Dataset root:', base_path)

In [ ]:
# ── Grouped-stratified split (anti-leakage) ────────────────────────────────
import re, shutil, random
from collections import defaultdict, Counter
from pathlib import Path

SEED = 42
random.seed(SEED)
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
TRAIN_RATIO, VAL_RATIO = 0.80, 0.10

SHRIMP_NAME_PATTERN = re.compile(
    r'^(?P<disease>Healthy|BG|WSSV_BG|WSSV)-(?P<shrimp_id>.+)-img-(?P<img_num>\d+)$',
    re.IGNORECASE,
)


def normalize_roboflow_stem(stem):
    stem = re.sub(r'_(jpg|jpeg|png|bmp|webp)\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    stem = re.sub(r'\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    return stem


def parse_shrimp_group_key(image_name):
    stem  = normalize_roboflow_stem(Path(image_name).stem)
    match = SHRIMP_NAME_PATTERN.match(stem)
    if not match:
        return f'unparsed::{stem}', 'unparsed', None, None
    disease  = match.group('disease')
    shrimp_id = match.group('shrimp_id')
    return f'{disease.lower()}::{shrimp_id}', disease, shrimp_id, int(match.group('img_num'))


def image_files_in_split(split):
    d = Path(base_path) / split / 'images'
    return sorted(p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def move_image_and_label(image_path, target_split):
    ti = Path(base_path) / target_split / 'images'
    tl = Path(base_path) / target_split / 'labels'
    ti.mkdir(parents=True, exist_ok=True)
    tl.mkdir(parents=True, exist_ok=True)
    lbl_src = image_path.parent.parent / 'labels' / f'{image_path.stem}.txt'
    img_dst = ti / image_path.name
    lbl_dst = tl / f'{image_path.stem}.txt'
    if image_path.resolve() != img_dst.resolve():
        shutil.move(str(image_path), str(img_dst))
    if lbl_src.exists():
        if lbl_src.resolve() != lbl_dst.resolve():
            shutil.move(str(lbl_src), str(lbl_dst))
    else:
        lbl_dst.write_text('')


def disease_for_group(filenames):
    diseases = [parse_shrimp_group_key(f)[1] for f in filenames]
    return Counter(diseases).most_common(1)[0][0]


def split_one_stratum(items):
    n = len(items)
    tr = int(TRAIN_RATIO * n)
    va = int(VAL_RATIO * n)
    te = n - tr - va
    if n >= 3:
        if va == 0: va, tr = 1, tr - 1
        if te == 0: te, tr = 1, tr - 1
    if tr < 1 and n > 0: tr = 1
    while tr + va + te > n: tr -= 1
    te = n - tr - va
    return items[:tr], items[tr:tr+va], items[tr+va:]


def remove_yolo_label_caches(root):
    for p in Path(root).glob('**/*.cache'): p.unlink()


# Rebuild pool from all splits then re-split
for split in ['train', 'valid', 'test']:
    for sub in ['images', 'labels']:
        os.makedirs(os.path.join(base_path, split, sub), exist_ok=True)

all_images = []
for split in ['train', 'valid', 'test']:
    all_images.extend(image_files_in_split(split))
for img in sorted(all_images):
    move_image_and_label(img, 'train')

groups = defaultdict(list)
for img in image_files_in_split('train'):
    gk, *_ = parse_shrimp_group_key(img.name)
    groups[gk].append(img.name)

strata = defaultdict(list)
for gk, filenames in sorted(groups.items()):
    strata[disease_for_group(filenames)].append((gk, filenames))

split_to_groups = {'train': [], 'valid': [], 'test': []}
rng = random.Random(SEED)
for disease, items in sorted(strata.items()):
    items = sorted(items, key=lambda x: x[0])
    rng.shuffle(items)
    tr, va, te = split_one_stratum(items)
    split_to_groups['train'].extend(tr)
    split_to_groups['valid'].extend(va)
    split_to_groups['test'].extend(te)
    print(f'  {disease}: {len(tr)} train, {len(va)} valid, {len(te)} test groups')

for split, sg in split_to_groups.items():
    for _, filenames in sg:
        for fn in filenames:
            move_image_and_label(Path(base_path) / 'train' / 'images' / fn, split)

# Leakage check
group_to_split = {}
for split in ['train', 'valid', 'test']:
    for ip in image_files_in_split(split):
        gk, *_ = parse_shrimp_group_key(ip.name)
        prev = group_to_split.setdefault(gk, split)
        if prev != split:
            raise RuntimeError(f'Leakage detected: {gk} in {prev} and {split}')
print('Shrimp-level leakage check PASSED.')
remove_yolo_label_caches(base_path)

for split in ['train', 'valid', 'test']:
    n = len(image_files_in_split(split))
    print(f'  {split}: {n} images')

In [ ]:
# ── Update data.yaml paths ─────────────────────────────────────────────────
import yaml

data_yaml_path = os.path.join(base_path, 'data.yaml')
with open(data_yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg['train'] = f'{base_path}/train/images'
cfg['val']   = f'{base_path}/valid/images'
cfg['test']  = f'{base_path}/test/images'

with open(data_yaml_path, 'w') as f:
    yaml.dump(cfg, f)

print('data.yaml updated.')
print('Classes:', cfg.get('names'))

In [ ]:
# ── Patch WIoU v3 into ultralytics loss.py ─────────────────────────────────
import torch
import ultralytics.utils.loss as ult_loss
from ultralytics.utils.metrics import bbox_iou

OrigBboxLoss = ult_loss.BboxLoss


class WIoUv3BboxLoss(OrigBboxLoss):
    """Drop-in replacement: replaces CIoU with WIoU v3 non-monotonic focusing."""

    WIOU_SCALE = 1.0

    def __call__(self, *args, **kwargs):
        # Use *args so signature is forward-compatible with any ultralytics version.
        # Positional order (all versions): pred_dist, pred_bboxes, anchor_points,
        #   target_bboxes, target_scores, target_scores_sum, fg_mask, [extra...]
        loss_iou, loss_dfl = super().__call__(*args, **kwargs)

        pred_bboxes       = args[1]
        target_bboxes     = args[3]
        target_scores     = args[4]
        target_scores_sum = args[5]
        fg_mask           = args[6]

        if fg_mask.sum() > 0:
            with torch.no_grad():
                iou = bbox_iou(
                    pred_bboxes[fg_mask],
                    target_bboxes[fg_mask],
                    xywh=False, CIoU=True,
                ).squeeze(-1).clamp(0, 1)

            beta  = (iou - 0.5).abs()
            alpha = 1.0 / (2.0 * beta + 1e-7)
            alpha = alpha / alpha.mean().clamp(min=1e-7)

            iou_full = bbox_iou(
                pred_bboxes[fg_mask],
                target_bboxes[fg_mask],
                xywh=False, CIoU=True,
            ).squeeze(-1)

            w = (target_scores[fg_mask].max(-1).values if target_scores.ndim > 1
                 else target_scores[fg_mask])
            wiou_loss = (alpha * (1.0 - iou_full) * w).sum() / target_scores_sum
            loss_iou  = wiou_loss * self.WIOU_SCALE

        return loss_iou, loss_dfl


ult_loss.BboxLoss = WIoUv3BboxLoss
print('WIoU v3 BboxLoss patch applied.')

In [ ]:
# ── Patch attention modules into ultralytics namespace ─────────────────────
# CoordAtt args in YAML: [c1, reduction] where c1 = ACTUAL scaled channels
# from the FROM layer (64/128/256 for YOLO11n with width_multiple=0.25).
# ultralytics passes YAML args directly for custom modules — c1 must match.

import torch
import torch.nn as nn
import ultralytics.nn.modules.conv as conv_module
import ultralytics.nn.modules as nn_modules
import ultralytics.nn.tasks as tasks_module


class SimAM(nn.Module):
    def __init__(self, c1=None, e_lambda=1e-4):
        super().__init__()
        self.e_lambda   = e_lambda
        self.activation = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()
        n = h * w - 1
        if n <= 0:
            return x
        x_mu  = x - x.mean(dim=[2, 3], keepdim=True)
        denom = 4 * (x_mu.pow(2).sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)
        y     = x_mu.pow(2) / denom + 0.5
        return x * self.activation(y)


class CoordAtt(nn.Module):
    # Channel-preserving: output channels == input channels (c1).
    # No separate c2 — ultralytics YAML args are (c1, reduction).
    def __init__(self, c1, reduction=32):
        super().__init__()
        mip = max(8, c1 // reduction)
        self.conv1  = nn.Conv2d(c1, mip, 1, 1, 0)
        self.bn1    = nn.BatchNorm2d(mip)
        self.act    = nn.SiLU()
        self.conv_h = nn.Conv2d(mip, c1, 1, 1, 0)
        self.conv_w = nn.Conv2d(mip, c1, 1, 1, 0)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        y   = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        a_h = self.conv_h(x_h).sigmoid()
        a_w = self.conv_w(x_w).sigmoid()
        return identity * a_h * a_w


for ns in [conv_module, nn_modules, tasks_module]:
    ns.SimAM    = SimAM
    ns.CoordAtt = CoordAtt
nn_modules.__all__ = list(set(list(getattr(nn_modules, '__all__', [])) + ['SimAM', 'CoordAtt']))
print('SimAM + CoordAtt registered in ultralytics namespace.')

In [ ]:
# ── Write SimAM+CA YAML to ultralytics cfg directory ───────────────────────
# CoordAtt args: [c1, reduction] where c1 = ACTUAL channels after width scaling.
# YOLO11n: width_multiple=0.25 → layer 16=64ch, layer 19=128ch, layer 22=256ch.
import ultralytics
from pathlib import Path

YAML_CONTENT = """
# YOLO11n-seg + SimAM + CA head — WIoU v3 experiment (Nhom B, B1)
nc: 2
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]

  - [16, 1, CoordAtt, [64, 32]]
  - [23, 1, SimAM, []]

  - [19, 1, CoordAtt, [128, 32]]
  - [25, 1, SimAM, []]

  - [22, 1, CoordAtt, [256, 32]]
  - [27, 1, SimAM, []]

  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]
""".strip()

ultralytics_root = Path(ultralytics.__file__).parent
yaml_dir  = ultralytics_root / 'cfg' / 'models' / '11'
yaml_dir.mkdir(parents=True, exist_ok=True)
yaml_path = yaml_dir / 'yolo11n-seg-simam-ca-wiouv3.yaml'
yaml_path.write_text(YAML_CONTENT)
print('YAML written to:', yaml_path)

In [ ]:
# ── Helper functions (reused from baseline) ────────────────────────────────
import gc, time, shutil
import yaml
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display

EXPERIMENT_ROOT = RUNTIME_ROOT / 'shrimp_wiouv3'
RUNS_DIR        = EXPERIMENT_ROOT / 'runs' / 'segment'
REPORT_DIR      = EXPERIMENT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

COUNT_PENALTY_WEIGHT      = 0.05
DISEASE_MISS_PENALTY_WEIGHT = 0.15
HEALTHY_FP_PENALTY_WEIGHT = 0.10
PREDICT_CONF_FOR_COUNT    = 0.25
TRAIN_IMGSZ = 640


def find_image_for_label(image_dir, label_name):
    stem = Path(label_name).stem
    for ext in IMAGE_EXTENSIONS:
        c = Path(image_dir) / f'{stem}{ext}'
        if c.exists():
            return c
    return None


def write_data_yaml(dataset_dir, out_yaml):
    with open(data_yaml_path) as f:
        c = yaml.safe_load(f)
    c['train'] = str(Path(dataset_dir) / 'train' / 'images')
    c['val']   = str(Path(dataset_dir) / 'valid' / 'images')
    c['test']  = str(Path(dataset_dir) / 'test'  / 'images')
    with open(out_yaml, 'w') as f:
        yaml.safe_dump(c, f, sort_keys=False)


def copy_dataset_for_experiment(exp_key):
    dst = EXPERIMENT_ROOT / exp_key / 'dataset'
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(base_path, dst, ignore=shutil.ignore_patterns('*.cache'))
    remove_yolo_label_caches(dst)
    return dst


def copy_split_by_label_state(src_dataset, dst_dataset, split, want_labeled):
    si = Path(src_dataset) / split / 'images'
    sl = Path(src_dataset) / split / 'labels'
    di = Path(dst_dataset) / split / 'images'
    dl = Path(dst_dataset) / split / 'labels'
    di.mkdir(parents=True, exist_ok=True)
    dl.mkdir(parents=True, exist_ok=True)
    copied = 0
    for lp in sorted(sl.glob('*.txt')):
        lines = [l.strip() for l in lp.read_text().splitlines() if l.strip()]
        if bool(lines) != want_labeled:
            continue
        ip = find_image_for_label(si, lp.name)
        if ip is None:
            continue
        shutil.copy2(ip, di / ip.name)
        shutil.copy2(lp, dl / lp.name)
        copied += 1
    return copied


def make_eval_dataset(src_dataset, exp_key, state_name, want_labeled):
    dst = EXPERIMENT_ROOT / exp_key / f'dataset_{state_name}_eval'
    if dst.exists():
        shutil.rmtree(dst)
    for sub in ['images', 'labels']:
        (dst / 'train' / sub).mkdir(parents=True, exist_ok=True)
    for split in ['valid', 'test']:
        copy_split_by_label_state(src_dataset, dst, split, want_labeled)
    yp = dst / f'data_{state_name}.yaml'
    write_data_yaml(dst, yp)
    return dst, yp


def metric_value(metrics, dotted_path, default=float('nan')):
    obj = metrics
    for part in dotted_path.split('.'):
        if not hasattr(obj, part):
            return default
        obj = getattr(obj, part)
    try:
        return float(obj)
    except Exception:
        return default


def count_prediction_errors(model, images_dir, labels_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = sorted(
        p for p in Path(images_dir).iterdir()
        if p.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        nan = float('nan')
        return dict(images=0, gt_total=0, pred_mask_total=0, mask_count_mae=nan,
                    disease_images=0, disease_box_miss_rate=nan, disease_mask_miss_rate=nan)
    results = model.predict(source=[str(p) for p in image_paths], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    mask_errs, disease_images, disease_box_miss = [], 0, 0
    gt_total = pred_total = 0
    for ip, res in zip(image_paths, results):
        lp = Path(labels_dir) / f'{ip.stem}.txt'
        gt = len([l for l in lp.read_text().splitlines() if l.strip()]) if lp.exists() else 0
        pm = len(res.masks) if res.masks is not None else 0
        pb = len(res.boxes) if res.boxes is not None else 0
        mask_errs.append(abs(pm - gt) / max(1, gt))
        if gt > 0:
            disease_images += 1
            disease_box_miss += int(pb == 0)
        gt_total += gt; pred_total += pm
    return dict(
        images=len(image_paths), gt_total=gt_total, pred_mask_total=pred_total,
        mask_count_mae=sum(mask_errs)/len(mask_errs) if mask_errs else float('nan'),
        disease_images=disease_images,
        disease_box_miss_rate=disease_box_miss/disease_images if disease_images else float('nan'),
        disease_mask_miss_rate=float('nan'),
    )


def healthy_false_positive_summary(model, images_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = sorted(
        p for p in Path(images_dir).iterdir()
        if p.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        nan = float('nan')
        return dict(healthy_images=0, healthy_mask_fp_rate=nan,
                    healthy_fp_masks_per_image=nan, healthy_avg_fp_confidence=0.0)
    results = model.predict(source=[str(p) for p in image_paths], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    with_fp = 0; mask_total = 0; confs = []
    for res in results:
        mc = len(res.masks) if res.masks is not None else 0
        if mc > 0:
            with_fp += 1
            mask_total += mc
            try:
                confs.extend(res.boxes.conf.cpu().tolist())
            except Exception:
                pass
    n = len(image_paths)
    return dict(
        healthy_images=n,
        healthy_mask_fp_rate=with_fp / n,
        healthy_fp_masks_per_image=mask_total / n,
        healthy_avg_fp_confidence=sum(confs)/len(confs) if confs else 0.0,
    )


def healthy_aware_score(labeled_map50, count_info, fp_info):
    return (labeled_map50
            - COUNT_PENALTY_WEIGHT * count_info['mask_count_mae']
            - DISEASE_MISS_PENALTY_WEIGHT * count_info['disease_box_miss_rate']
            - HEALTHY_FP_PENALTY_WEIGHT * fp_info['healthy_mask_fp_rate'])


def read_best_epoch_from_results(run_path):
    csv_path = Path(run_path) / 'results.csv'
    if not csv_path.exists():
        return {}
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    mc = 'metrics/mAP50(M)'
    if mc not in df.columns:
        return {'epochs_ran': len(df)}
    bi = df[mc].idxmax()
    best = df.iloc[bi]
    last = df.iloc[-1]
    return dict(
        epochs_ran=len(df),
        best_epoch_by_mask_map50=int(best.get('epoch', bi + 1)),
        best_val_mask_map50=float(best.get(mc, float('nan'))),
        last_train_seg_loss=float(last.get('train/seg_loss', float('nan'))),
        last_val_seg_loss=float(last.get('val/seg_loss', float('nan'))),
        seg_loss_gap_val_minus_train=float(last.get('val/seg_loss', 0) - last.get('train/seg_loss', 0)),
    )


def disable_ultralytics_albumentations():
    try:
        import ultralytics.data.augment as aug
        class _NoOp:
            contains_spatial = False
            def __init__(self, *a, **kw): self.transform = None
            def __call__(self, labels): return labels
        aug.Albumentations = _NoOp
    except Exception as e:
        print('Albumentations patch skipped:', e)


print('Helper functions defined.')

In [ ]:
# ── Training — SimAM+CA + WIoU v3 ─────────────────────────────────────────
CLEAN_TRAIN_ARGS = dict(
    auto_augment=None,
    erasing=0.0, mosaic=0.0, mixup=0.0, cutmix=0.0, copy_paste=0.0,
    fliplr=0.5, flipud=0.0,
    hsv_h=0.01, hsv_s=0.35, hsv_v=0.20,
    degrees=0.0, translate=0.05, scale=0.20,
    shear=0.0, perspective=0.0, multi_scale=0.0, bgr=0.0,
)

exp_key  = 'simam_ca_wiouv3'
run_name = f'yolo11n-seg_{exp_key}'

dataset_dir = copy_dataset_for_experiment(exp_key)
exp_yaml    = dataset_dir / 'data.yaml'
write_data_yaml(dataset_dir, exp_yaml)

labeled_eval_dir, labeled_eval_yaml = make_eval_dataset(dataset_dir, exp_key, 'labeled_only', True)
healthy_eval_dir, healthy_eval_yaml = make_eval_dataset(dataset_dir, exp_key, 'healthy_only', False)

disable_ultralytics_albumentations()

# Build model from YAML and load pretrained backbone weights
yolo = YOLO(str(yaml_path))
try:
    yolo.load('yolo11n-seg.pt')
except Exception as e:
    print('Pretrained load warning:', e)

start = time.time()
yolo.train(
    data=str(exp_yaml),
    task='segment',
    imgsz=TRAIN_IMGSZ,
    epochs=100,
    batch=16,
    patience=30,
    seed=42,
    deterministic=True,
    workers=0,
    project=str(RUNS_DIR),
    name=run_name,
    exist_ok=True,
    pretrained=True,
    plots=True,
    verbose=True,
    **CLEAN_TRAIN_ARGS,
)
train_time_min = (time.time() - start) / 60
print(f'Training finished in {train_time_min:.1f} min.')

In [ ]:
# ── Evaluation ─────────────────────────────────────────────────────────────
run_path   = RUNS_DIR / run_name
best_path  = run_path / 'weights' / 'best.pt'
best_model = YOLO(str(best_path))

full_val       = best_model.val(data=str(exp_yaml),          split='val',  imgsz=TRAIN_IMGSZ, plots=True,  verbose=False)
full_test      = best_model.val(data=str(exp_yaml),          split='test', imgsz=TRAIN_IMGSZ, plots=True,  verbose=False)
labeled_val    = best_model.val(data=str(labeled_eval_yaml), split='val',  imgsz=TRAIN_IMGSZ, plots=False, verbose=False)
labeled_test   = best_model.val(data=str(labeled_eval_yaml), split='test', imgsz=TRAIN_IMGSZ, plots=False, verbose=False)

labeled_test_count = count_prediction_errors(
    best_model, labeled_eval_dir/'test'/'images', labeled_eval_dir/'test'/'labels')
healthy_test_fp    = healthy_false_positive_summary(
    best_model, healthy_eval_dir/'test'/'images')

labeled_test_map50 = metric_value(labeled_test, 'seg.map50')
h_score = healthy_aware_score(labeled_test_map50, labeled_test_count, healthy_test_fp)

row = dict(
    experiment=exp_key,
    loss_variant='WIoU_v3',
    train_time_min=round(train_time_min, 2),
    full_val_mask_map50=metric_value(full_val,  'seg.map50'),
    full_test_mask_map50=metric_value(full_test, 'seg.map50'),
    labeled_val_mask_map50=metric_value(labeled_val,  'seg.map50'),
    labeled_test_mask_map50=labeled_test_map50,
    labeled_test_mask_map50_95=metric_value(labeled_test, 'seg.map'),
    healthy_test_mask_fp_rate=healthy_test_fp['healthy_mask_fp_rate'],
    healthy_aware_score=h_score,
    **labeled_test_count,
)
row.update(read_best_epoch_from_results(run_path))

results_df = pd.DataFrame([row])
results_df.to_csv(REPORT_DIR / 'wiouv3_results.csv', index=False)
display(results_df.T)

del best_model; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# ── Visualise training curves ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_png = run_path / 'results.png'
if results_png.exists():
    plt.figure(figsize=(14, 10))
    plt.imshow(mpimg.imread(results_png))
    plt.axis('off')
    plt.title('WIoU v3 Training Curves')
    plt.show()
else:
    print('results.png not found at', results_png)

## Test-time augmentation inference — simam_ca_wiouv3

Run only after leakage-safe splitting and training. Each original test image is inferred in six geometric views; predictions are inverse-transformed, instance-matched, voted, mask-fused, and evaluated against the original labeled test split. No augmented test files are created.


In [ ]:
# =========================================================
# Paper-style instance segmentation TTA on the isolated test split
# =========================================================
from pathlib import Path
from collections import defaultdict
import json

import cv2
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from ultralytics import YOLO

TTA_CONFIG = {
    "enabled": True,
    "imgsz": 640,  # project adaptation; paper used 512
    "transforms": [
        "original", "horizontal_flip", "vertical_flip",
        "rotate_90", "rotate_180", "rotate_270",
    ],
    "inverse_boxes": True,
    "inverse_masks": True,
    "matching_metric": "mask_iou",
    "matching_iou_threshold": 0.50,
    "matching_assignment": "greedy_best_match",
    "one_to_one": True,
    "search_unmatched_views": True,
    "minimum_support": 4,
    "total_views": 6,
    "mask_fusion": "average_binary_masks",
    "mask_threshold": 0.50,
    "conf": 0.25,
    "iou_nms": 0.70,
    "max_det": 300,
    "agnostic_nms": False,
    "class_aware_matching": True,
    "final_mask_nms_iou": 0.50,
}

assert len(TTA_CONFIG["transforms"]) == TTA_CONFIG["total_views"]
assert TTA_CONFIG["minimum_support"] > TTA_CONFIG["total_views"] / 2


def apply_view(image, name):
    if name == "original": return image.copy()
    if name == "horizontal_flip": return cv2.flip(image, 1)
    if name == "vertical_flip": return cv2.flip(image, 0)
    if name == "rotate_90": return cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    if name == "rotate_180": return cv2.rotate(image, cv2.ROTATE_180)
    if name == "rotate_270": return cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
    raise ValueError(name)


def inverse_mask(mask, name, original_hw):
    # OpenCV flip/rotate do not accept numpy bool arrays (notably OpenCV 4.10).
    mask = np.ascontiguousarray(mask, dtype=np.uint8)
    if name == "horizontal_flip": mask = cv2.flip(mask, 1)
    elif name == "vertical_flip": mask = cv2.flip(mask, 0)
    elif name == "rotate_90": mask = cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif name == "rotate_180": mask = cv2.rotate(mask, cv2.ROTATE_180)
    elif name == "rotate_270": mask = cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    elif name != "original": raise ValueError(name)
    h, w = original_hw
    mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    return mask > 0


def mask_iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter / union) if union else 0.0


def predict_view(model, image, view_name, original_hw):
    transformed = apply_view(image, view_name)
    result = model.predict(
        source=transformed, imgsz=TTA_CONFIG["imgsz"], conf=TTA_CONFIG["conf"],
        iou=TTA_CONFIG["iou_nms"], max_det=TTA_CONFIG["max_det"],
        agnostic_nms=TTA_CONFIG["agnostic_nms"], verbose=False,
    )[0]
    if result.masks is None or result.boxes is None:
        return []
    raw_masks = result.masks.data.detach().cpu().numpy()
    classes = result.boxes.cls.detach().cpu().numpy().astype(int)
    scores = result.boxes.conf.detach().cpu().numpy()
    vh, vw = transformed.shape[:2]
    instances = []
    for raw_mask, cls_id, score in zip(raw_masks, classes, scores):
        view_mask = cv2.resize(raw_mask, (vw, vh), interpolation=cv2.INTER_NEAREST) >= 0.5
        instances.append({
            "mask": inverse_mask(view_mask, view_name, original_hw),
            "class_id": int(cls_id), "confidence": float(score), "view": view_name,
        })
    return instances


def fuse_views(view_predictions):
    tracks = []
    for view_name in TTA_CONFIG["transforms"]:
        instances = view_predictions.get(view_name, [])
        candidates = []
        for ti, track in enumerate(tracks):
            reference = np.mean([x["mask"] for x in track], axis=0) >= TTA_CONFIG["mask_threshold"]
            for ii, inst in enumerate(instances):
                if TTA_CONFIG["class_aware_matching"] and track[0]["class_id"] != inst["class_id"]:
                    continue
                iou = mask_iou(reference, inst["mask"])
                if iou >= TTA_CONFIG["matching_iou_threshold"]:
                    candidates.append((iou, ti, ii))
        used_tracks, used_instances = set(), set()
        for _, ti, ii in sorted(candidates, reverse=True):
            if ti in used_tracks or ii in used_instances: continue
            tracks[ti].append(instances[ii]); used_tracks.add(ti); used_instances.add(ii)
        if TTA_CONFIG["search_unmatched_views"]:
            tracks.extend([[inst] for ii, inst in enumerate(instances) if ii not in used_instances])

    fused = []
    for track in tracks:
        if len(track) < TTA_CONFIG["minimum_support"]: continue
        fused_mask = np.mean([x["mask"] for x in track], axis=0) >= TTA_CONFIG["mask_threshold"]
        if not fused_mask.any(): continue
        fused.append({
            "mask": fused_mask, "class_id": track[0]["class_id"],
            "confidence": float(np.mean([x["confidence"] for x in track])),
            "support": len(track),
        })

    kept = []
    for pred in sorted(fused, key=lambda x: x["confidence"], reverse=True):
        suppress = any(
            (not TTA_CONFIG["class_aware_matching"] or pred["class_id"] == old["class_id"])
            and mask_iou(pred["mask"], old["mask"]) >= TTA_CONFIG["final_mask_nms_iou"]
            for old in kept
        )
        if not suppress: kept.append(pred)
    return kept


def load_gt_masks(label_path, hw):
    h, w = hw; output = []
    if not label_path.exists(): return output
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 7: continue
        cls_id = int(parts[0])
        xy = np.asarray(parts[1:], dtype=float).reshape(-1, 2)
        xy[:, 0] *= w; xy[:, 1] *= h
        canvas = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(canvas, [np.round(xy).astype(np.int32)], 1)
        output.append({"class_id": cls_id, "mask": canvas.astype(bool)})
    return output


def ap_from_pr(rec, prec):
    mrec = np.r_[0.0, rec, 1.0]; mpre = np.r_[1.0, prec, 0.0]
    mpre = np.maximum.accumulate(mpre[::-1])[::-1]
    return float(np.trapz(np.interp(np.linspace(0, 1, 101), mrec, mpre), np.linspace(0, 1, 101)))


def labeled_mask_map50(records, class_names):
    rows = []
    for cls_id, class_name in enumerate(class_names):
        total_gt = sum(sum(g["class_id"] == cls_id for g in r["gt"]) for r in records)
        ranked = []
        for image_id, r in enumerate(records):
            ranked.extend((p["confidence"], image_id, p) for p in r["pred"] if p["class_id"] == cls_id)
        ranked.sort(reverse=True, key=lambda x: x[0]); matched = defaultdict(set); tp=[]; fp=[]
        for _, image_id, pred in ranked:
            gts = records[image_id]["gt"]
            options = [(mask_iou(pred["mask"], g["mask"]), gi) for gi, g in enumerate(gts)
                       if g["class_id"] == cls_id and gi not in matched[image_id]]
            best_iou, best_gi = max(options, default=(0.0, -1))
            hit = best_iou >= 0.50
            tp.append(float(hit)); fp.append(float(not hit))
            if hit: matched[image_id].add(best_gi)
        if total_gt:
            tp_c = np.cumsum(tp); fp_c = np.cumsum(fp)
            rec = tp_c / total_gt; prec = tp_c / np.maximum(tp_c + fp_c, 1e-12)
            ap50 = ap_from_pr(rec, prec)
        else: ap50 = np.nan
        rows.append({"class_id": cls_id, "class": class_name, "gt_instances": total_gt,
                     "pred_instances": len(ranked), "mask_AP50": ap50})
    frame = pd.DataFrame(rows)
    return frame, float(frame["mask_AP50"].dropna().mean())


def draw_fused(image, predictions, class_names):
    canvas = image.copy(); rng = np.random.default_rng(42)
    colors = {i: tuple(int(x) for x in rng.integers(40, 256, 3)) for i in range(len(class_names))}
    for pred in predictions:
        color = colors.get(pred["class_id"], (0, 255, 0)); mask = pred["mask"]
        overlay = np.zeros_like(canvas); overlay[mask] = color
        canvas = cv2.addWeighted(canvas, 1.0, overlay, 0.40, 0)
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(canvas, contours, -1, color, 2)
        if contours:
            x, y, _, _ = cv2.boundingRect(max(contours, key=cv2.contourArea))
            label = f'{class_names[pred["class_id"]]} {pred["confidence"]:.2f} ({pred["support"]}/6)'
            cv2.putText(canvas, label, (x, max(y-5, 15)), cv2.FONT_HERSHEY_SIMPLEX, .45, color, 1, cv2.LINE_AA)
    return canvas



def mask_to_xyxy(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return np.asarray([0.0, 0.0, 0.0, 0.0], dtype=float)
    return np.asarray([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1], dtype=float)


def box_iou_xyxy(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    union = area_a + area_b - inter
    return float(inter / union) if union > 0 else 0.0


def evaluate_one_iou(records, class_id, iou_threshold, metric_type):
    total_gt = sum(sum(g["class_id"] == class_id for g in record["gt"]) for record in records)
    ranked = []
    for image_id, record in enumerate(records):
        ranked.extend(
            (pred["confidence"], image_id, pred)
            for pred in record["pred"] if pred["class_id"] == class_id
        )
    ranked.sort(key=lambda item: item[0], reverse=True)
    matched = defaultdict(set)
    tp, fp = [], []
    for _, image_id, pred in ranked:
        candidates = []
        for gt_id, gt in enumerate(records[image_id]["gt"]):
            if gt["class_id"] != class_id or gt_id in matched[image_id]:
                continue
            if metric_type == "mask":
                overlap = mask_iou(pred["mask"], gt["mask"])
            else:
                overlap = box_iou_xyxy(mask_to_xyxy(pred["mask"]), mask_to_xyxy(gt["mask"]))
            candidates.append((overlap, gt_id))
        best_iou, best_gt = max(candidates, default=(0.0, -1))
        is_tp = best_iou >= iou_threshold
        tp.append(float(is_tp)); fp.append(float(not is_tp))
        if is_tp:
            matched[image_id].add(best_gt)

    tp_sum, fp_sum = float(sum(tp)), float(sum(fp))
    precision = tp_sum / max(tp_sum + fp_sum, 1e-12)
    recall = tp_sum / max(total_gt, 1e-12)
    if total_gt and ranked:
        tp_curve, fp_curve = np.cumsum(tp), np.cumsum(fp)
        recall_curve = tp_curve / total_gt
        precision_curve = tp_curve / np.maximum(tp_curve + fp_curve, 1e-12)
        ap = ap_from_pr(recall_curve, precision_curve)
    else:
        ap = 0.0 if total_gt else np.nan
    return {"precision": precision, "recall": recall, "ap": ap,
            "gt": total_gt, "pred": len(ranked)}


def evaluate_tta_like_ultralytics(records, class_names):
    """Return YOLO-style P, R, mAP50 and mAP50-95 for fused boxes and masks."""
    iou_thresholds = np.arange(0.50, 0.96, 0.05)
    class_rows = []
    for class_id, class_name in enumerate(class_names):
        row = {"class_id": class_id, "class": class_name}
        for metric_type, suffix in (("box", "B"), ("mask", "M")):
            evaluations = [
                evaluate_one_iou(records, class_id, float(threshold), metric_type)
                for threshold in iou_thresholds
            ]
            at_50 = evaluations[0]
            row.update({
                f"precision_{suffix}": at_50["precision"],
                f"recall_{suffix}": at_50["recall"],
                f"mAP50_{suffix}": at_50["ap"],
                f"mAP50-95_{suffix}": float(np.nanmean([item["ap"] for item in evaluations])),
            })
            if metric_type == "mask":
                row["gt_instances"] = at_50["gt"]
                row["pred_instances"] = at_50["pred"]
        class_rows.append(row)

    per_class = pd.DataFrame(class_rows)
    valid = per_class[per_class["gt_instances"] > 0]
    if valid.empty:
        raise RuntimeError("No labeled ground-truth instances were found in the test split")
    overall = {
        "metrics/precision(B)": float(valid["precision_B"].mean()),
        "metrics/recall(B)": float(valid["recall_B"].mean()),
        "metrics/mAP50(B)": float(valid["mAP50_B"].mean()),
        "metrics/mAP50-95(B)": float(valid["mAP50-95_B"].mean()),
        "metrics/precision(M)": float(valid["precision_M"].mean()),
        "metrics/recall(M)": float(valid["recall_M"].mean()),
        "metrics/mAP50(M)": float(valid["mAP50_M"].mean()),
        "metrics/mAP50-95(M)": float(valid["mAP50-95_M"].mean()),
    }
    return overall, per_class


def resolve_tta_checkpoint(experiment_keys=(), direct_variable_names=()):
    """Resolve the requested trained checkpoint without assuming one summary variable name."""
    experiment_keys = tuple(str(key) for key in experiment_keys)

    # Prefer an exact experiment match in any summary DataFrame created by training.
    for frame_name in ("summary_df", "results_df", "summary_for_test_aug"):
        frame = globals().get(frame_name)
        if not isinstance(frame, pd.DataFrame) or frame.empty or "best_pt" not in frame.columns:
            continue
        key_column = next((column for column in ("experiment", "key", "experiment_key") if column in frame.columns), None)
        if key_column and experiment_keys:
            matched = frame[frame[key_column].astype(str).isin(experiment_keys)]
            for value in matched["best_pt"].tolist():
                candidate = Path(value)
                if candidate.exists():
                    print(f"TTA checkpoint from {frame_name}: {candidate}")
                    return candidate

    # Single-experiment notebooks expose one of these path variables after training.
    for variable_name in direct_variable_names:
        value = globals().get(variable_name)
        if value is None:
            continue
        candidate = Path(value)
        if candidate.exists():
            print(f"TTA checkpoint from {variable_name}: {candidate}")
            return candidate

    # Last fallback: search runtime outputs, preferring paths containing the experiment key.
    search_roots = []
    for variable_name in ("EXPERIMENT_ROOT", "RUNS_DIR", "RUNTIME_ROOT"):
        value = globals().get(variable_name)
        if value is not None:
            root = Path(value)
            if root.exists() and root not in search_roots:
                search_roots.append(root)
    candidates = []
    for root in search_roots:
        candidates.extend(root.rglob("best.pt"))
    if experiment_keys:
        exact_candidates = [
            path for path in candidates
            if any(key.lower() in str(path).lower() for key in experiment_keys)
        ]
        if exact_candidates:
            candidates = exact_candidates
    if candidates:
        candidate = max(set(candidates), key=lambda path: path.stat().st_mtime)
        print(f"TTA checkpoint discovered from runtime outputs: {candidate}")
        return candidate

    raise FileNotFoundError(
        "Could not resolve a trained best.pt. Run the training/evaluation cell first. "
        f"Requested experiments={experiment_keys}, checked variables={direct_variable_names}."
    )


tta_checkpoint = resolve_tta_checkpoint(
    experiment_keys=("simam_ca_wiouv3", "wiouv3"),
    direct_variable_names=("best_path", "best_model_path"),
)
if not tta_checkpoint.exists(): raise FileNotFoundError(tta_checkpoint)
tta_model = YOLO(str(tta_checkpoint))

source_dataset = Path(base_path).resolve()
test_images = source_dataset / "test" / "images"
test_labels = source_dataset / "test" / "labels"
data_cfg = yaml.safe_load(Path(data_yaml_path).read_text())
names_obj = data_cfg.get("names", tta_model.names)
class_names = [names_obj[k] for k in sorted(names_obj)] if isinstance(names_obj, dict) else list(names_obj)
image_paths = sorted(p for p in test_images.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
if not image_paths: raise RuntimeError(f"No test images found in {test_images}")

records = []
for index, image_path in enumerate(image_paths, 1):
    image = cv2.imread(str(image_path))
    if image is None: raise FileNotFoundError(image_path)
    view_predictions = {name: predict_view(tta_model, image, name, image.shape[:2]) for name in TTA_CONFIG["transforms"]}
    fused = fuse_views(view_predictions)
    gt = load_gt_masks(test_labels / f"{image_path.stem}.txt", image.shape[:2])
    records.append({"path": image_path, "image": image, "pred": fused, "gt": gt})
    if index % 10 == 0 or index == len(image_paths): print(f"TTA: {index}/{len(image_paths)} images")

per_class_tta, labeled_mask_map50_value = labeled_mask_map50(records, class_names)
tta_metrics, tta_metrics_per_class = evaluate_tta_like_ultralytics(records, class_names)

print("\nTTA fused metrics on the original labeled test split")
print(pd.DataFrame([tta_metrics]).to_string(index=False))
print("\nPer-class TTA metrics")
display(tta_metrics_per_class)

output_dir = RUNTIME_ROOT / "tta_inference" / "simam_ca_wiouv3"
output_dir.mkdir(parents=True, exist_ok=True)
per_class_tta.to_csv(output_dir / "tta_mask_ap50_per_class.csv", index=False)
tta_metrics_per_class.to_csv(output_dir / "tta_metrics_per_class.csv", index=False)
pd.DataFrame([tta_metrics]).to_csv(output_dir / "tta_metrics_summary.csv", index=False)
(output_dir / "tta_config.json").write_text(json.dumps(TTA_CONFIG, indent=2))
(output_dir / "tta_summary.json").write_text(json.dumps({
    "checkpoint": str(tta_checkpoint),
    "test_images": len(records),
    **tta_metrics,
}, indent=2))

preview_count = min(12, len(records))
fig, axes = plt.subplots((preview_count + 2)//3, 3, figsize=(18, 6*((preview_count+2)//3)))
axes = np.asarray(axes).reshape(-1)
for ax, record in zip(axes, records[:preview_count]):
    rendered = draw_fused(record["image"], record["pred"], class_names)
    ax.imshow(cv2.cvtColor(rendered, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{record["path"].name} | fused={len(record["pred"])}')
    ax.axis("off")
for ax in axes[preview_count:]: ax.axis("off")
plt.tight_layout(); plt.show()
print(f"TTA outputs: {output_dir}")
